# create coarsened global maps

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import coiled
import cartopy
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import numpy as np
import zarr

In [2]:
config = Config('config/global_config_v9.txt')

----------------------------------------
Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
seasonal_snow_mask_reproject_method = rasterio
-------------------

/home/eric/repos/global_snowmelt_runoff_onset/global_snowmelt_runoff_onset/config.py:299: UserWarning: Could not parse SAS token expiration date: SAS token is EXPIRED! Expired 0.2 hours ago on 2025-10-14 16:57 UTC
  warnings.warn(f"Could not parse SAS token expiration date: {e}")


In [38]:
global_ds = xr.open_zarr(config.global_runoff_store, 
                        consolidated=True, 
                        decode_coords='all',
                        chunks={"longitude":8640, "latitude":4320}, #config.chunks_zarr_output,
                        )


global_ds

/tmp/ipykernel_3845452/2794366721.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 4320. This could degrade performance. Instead, consider rechunking after loading.
  global_ds = xr.open_zarr(config.global_runoff_store,
/tmp/ipykernel_3845452/2794366721.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 8640. This could degrade performance. Instead, consider rechunking after loading.
  global_ds = xr.open_zarr(config.global_runoff_store,


<xarray.Dataset> Size: 14TB
Dimensions:                     (latitude: 195970, longitude: 499998,
                                 water_year: 10)
Coordinates:
  * latitude                    (latitude) float64 2MB 81.1 81.1 ... -60.0 -60.0
  * longitude                   (longitude) float64 4MB -180.0 -180.0 ... 180.0
    spatial_ref                 int32 4B ...
  * water_year                  (water_year) int64 80B 2015 2016 ... 2023 2024
Data variables:
    runoff_onset                (water_year, latitude, longitude) float32 4TB dask.array<chunksize=(1, 4320, 8640), meta=np.ndarray>
    runoff_onset_mad            (latitude, longitude) float64 784GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
    runoff_onset_median         (latitude, longitude) float32 392GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
    temporal_resolution         (water_year, latitude, longitude) float64 8TB dask.array<chunksize=(1, 4320, 8640), meta=np.ndarray>
    temporal_resolution_median  (latitude, longitude) float64 784GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
Attributes:
    processed_tiles:  []

In [39]:
global_ds['runoff_onset_mad'] = global_ds['runoff_onset_mad'].astype('float32')
global_ds['temporal_resolution'] = global_ds['temporal_resolution'].astype('float32')
global_ds['temporal_resolution_median'] = global_ds['temporal_resolution_median'].astype('float32')

global_ds

<xarray.Dataset> Size: 9TB
Dimensions:                     (latitude: 195970, longitude: 499998,
                                 water_year: 10)
Coordinates:
  * latitude                    (latitude) float64 2MB 81.1 81.1 ... -60.0 -60.0
  * longitude                   (longitude) float64 4MB -180.0 -180.0 ... 180.0
    spatial_ref                 int32 4B ...
  * water_year                  (water_year) int64 80B 2015 2016 ... 2023 2024
Data variables:
    runoff_onset                (water_year, latitude, longitude) float32 4TB dask.array<chunksize=(1, 4320, 8640), meta=np.ndarray>
    runoff_onset_mad            (latitude, longitude) float32 392GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
    runoff_onset_median         (latitude, longitude) float32 392GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
    temporal_resolution         (water_year, latitude, longitude) float32 4TB dask.array<chunksize=(1, 4320, 8640), meta=np.ndarray>
    temporal_resolution_median  (latitude, longitude) float32 392GB dask.array<chunksize=(4320, 8640), meta=np.ndarray>
Attributes:
    processed_tiles:  []

In [ ]:
coarsen_factor = 20

global_coarsened_ds = global_ds.coarsen(
    latitude=coarsen_factor,
    longitude=coarsen_factor,
    
    boundary='trim').mean(
)
    
global_coarsened_ds

In [ ]:
@coiled.function(
    name="coarsen_global_dataset",
    memory="32 GB",
    #cpu=4,
    n_workers=30,  # Adaptive scaling within each function
    idle_timeout="30 minutes",
    keepalive="5 minutes",  # Keep VMs warm between batches
    spot_policy="spot",
    environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
    #region="westeurope",
    workspace="uwtacolab",
)
def coarsen_global_ds_and_save(config,coarsen_factor):
    store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_{config.version}_coarsened_{coarsen_factor}_ds.zarr")

    chunk_size = config.spatial_chunk_dims_zarr
    nodata_int16 = -9999
    encoding = {
        "runoff_onset": {
            "chunks": (1,) + chunk_size,
            "compressor": zarr.Blosc(cname="zstd"),
            "_FillValue": nodata_int16,
            "dtype": "int16",
        },
        "runoff_onset_median": {
            "chunks": chunk_size,
            "compressor": zarr.Blosc(cname="zstd"),
            "_FillValue": nodata_int16,
            "dtype": "int16",
        },
        "runoff_onset_mad": {
            "chunks": chunk_size,
            "compressor": zarr.Blosc(cname="zstd"),
            "_FillValue": nodata_int16,
            "dtype": "int16",
            "scale_factor": np.float32(0.1),
            "add_offset": np.float32(0.0),
        },
        "temporal_resolution_median": {
            "chunks": chunk_size,
            "compressor": zarr.Blosc(cname="zstd"),
            "_FillValue": nodata_int16,
            "dtype": "int16",
            "scale_factor": np.float32(0.1),
            "add_offset": np.float32(0.0),
        },
        "temporal_resolution": {
            "chunks": (1,) + chunk_size,
            "compressor": zarr.Blosc(cname="zstd"),
            "_FillValue": nodata_int16,
            "dtype": "int16",
            "scale_factor": np.float32(0.1),
            "add_offset": np.float32(0.0),
        },
    }

    global_ds = xr.open_zarr(config.global_runoff_store, 
                            consolidated=True, 
                            decode_coords='all',
                            chunks={"longitude":8640, "latitude":4320}, #config.chunks_zarr_output,
                            )


    global_ds['runoff_onset_mad'] = global_ds['runoff_onset_mad'].astype('float32')
    global_ds['temporal_resolution'] = global_ds['temporal_resolution'].astype('float32')
    global_ds['temporal_resolution_median'] = global_ds['temporal_resolution_median'].astype('float32')

    global_ds.coarsen(
        latitude=coarsen_factor,
        longitude=coarsen_factor,
        
        boundary='trim').mean(
    ).chunk(config.chunks_zarr_output).to_zarr( # take out chunk before zarr \.chunk(config.chunks_zarr_output)
            store=store,
            compute=True, 
            encoding=encoding,
            mode='w')

In [ ]:
coarsen_factor = 20

coarsen_global_ds_and_save(config, coarsen_factor)